# Why C++? Why OOP?

You already know C. You can write functions, use pointers, manage memory with `malloc`/`free`, and build reasonably complex programs. So why learn C++?

This notebook answers that question. We will look at what C is missing, what C++ adds, and why **Object-Oriented Programming (OOP)** is a valuable way to structure large programs. By the end you will have written your first C++ class.

## The Limits of C

C is a powerful, low-level language — but it was designed in the 1970s for systems programming, not for building large applications with many developers. As programs grew larger, C's limitations became painful:

- **No classes or encapsulation.** Data and the functions that operate on it live separately. Nothing stops you from accidentally modifying data you shouldn't touch.
- **No function overloading.** You can't have two functions named `print` — one for `int` and one for `double`. You need `printInt` and `printDouble`.
- **No RAII.** Resource management (files, memory, sockets) must be done manually. Forget a `free()` or `fclose()` and you have a leak.
- **No namespaces.** Every function and global name lives in one flat namespace. Name collisions across libraries are common.
- **No references.** You must use pointers to pass by address, which adds syntactic noise and risks null dereference.
- **No standard string type.** `char` arrays are error-prone; `strcat`, `strcpy`, and friends cause buffer overflows.

The code below shows the C-style approach to managing a simple "person" data structure in C++ syntax. Notice how verbose and fragile it is.

In [ ]:
#include <iostream>
#include <cstring>
#include <cstdlib>

// C-style: data and functions are completely separate
struct Person {
    char name[64];
    int  age;
};

// Must pass the struct explicitly every time
void personInit(Person *p, const char *name, int age) {
    strncpy(p->name, name, 63);
    p->name[63] = '\0';  // always null-terminate manually
    p->age = age;
}

void personPrint(const Person *p) {
    std::cout << p->name << ", age " << p->age << std::endl;
}

Person alice;
personInit(&alice, "Alice", 30);
personPrint(&alice);

// Nothing prevents this accidental corruption:
alice.age = -999;  // no protection!
personPrint(&alice);
// Expected output:
// Alice, age 30
// Alice, age -999

## What is OOP?

**Object-Oriented Programming** is a programming paradigm that organises code around *objects* — data bundled together with the functions that operate on it.

OOP is built on four pillars:

### 1. Encapsulation
Bundle data and behaviour together, and hide internal details. A `BankAccount` object knows its `balance` but doesn't let external code set it to an arbitrary value — it only exposes `deposit()` and `withdraw()`. **Analogy:** A vending machine — you press a button and get a snack; you don't need to know (or touch) the internal mechanics.

### 2. Inheritance
A new class can *inherit* the properties and behaviour of an existing class and extend or specialise them. A `Dog` is an `Animal` — it shares `eat()` and `sleep()` but also has `bark()`. **Analogy:** A sports car is a car — it has everything a car has, plus extra performance features.

### 3. Polymorphism
The same function call can behave differently depending on the actual type of the object. You can call `makeSound()` on any `Animal` and each subclass (Dog, Cat, Bird) responds appropriately. **Analogy:** A power outlet — you plug in a phone charger, a laptop, or a lamp; the interface is the same but each device uses electricity differently.

### 4. Abstraction
Show only the essential features, hide the complexity. You drive a car using a steering wheel and pedals without knowing how the engine works. **Analogy:** A TV remote — you press "Volume Up" without knowing how the infrared signal is encoded.

## Features of C++

C++ was designed by Bjarne Stroustrup starting in 1979 as "C with Classes". It adds the following to C:

| Feature | Description |
|---|---|
| **Classes** | Bundle data and functions; support encapsulation, inheritance, polymorphism |
| **Function overloading** | Multiple functions with the same name but different parameter types |
| **Operator overloading** | Define what `+`, `<<`, `==`, etc. mean for your types |
| **References** | An alias for a variable — cleaner than pointers for most use cases |
| **Namespaces** | Group names to avoid collisions (`std::`, `mylib::`) |
| **const correctness** | Enforce immutability through the type system |
| **RAII** | Constructors acquire resources; destructors release them automatically |
| **std::string** | A safe, resizable string class — no more fixed buffers |
| **Templates** | Generic programming — write code that works for any type |
| **Standard Library** | Containers (vector, map, list), algorithms, I/O streams, and more |

The cell below shows a before/after comparison using `std::string` instead of `char` arrays.

In [ ]:
#include <iostream>
#include <string>

// --- C-style (before) ---
char cName[64];
strncpy(cName, "Alice", 63);
// Concatenation is error-prone:
// strncat(cName, " Smith", 63 - strlen(cName));

// --- C++ style (after) ---
std::string cppName = "Alice";
cppName += " Smith";  // safe, automatic resizing

std::cout << "C-style name:   " << cName   << std::endl;
std::cout << "C++ style name: " << cppName << std::endl;
std::cout << "Length: " << cppName.length() << std::endl;
// Expected output:
// C-style name:   Alice
// C++ style name: Alice Smith
// Length: 11

## A First Class

Let us rewrite the `Person` example from the start of this notebook as a proper C++ class.

Key ideas:
- `private:` members can only be accessed from inside the class.
- `public:` members form the interface that external code can use.
- The **constructor** (same name as the class, no return type) is called automatically when an object is created.
- **Getter methods** provide controlled read access to private data.

In [ ]:
#include <iostream>
#include <string>

class Person {
private:
    std::string name;
    int age;

public:
    // Constructor
    Person(const std::string &n, int a) {
        name = n;
        age  = a;
    }

    // Getters
    std::string getName() const { return name; }
    int         getAge()  const { return age;  }

    // Method
    void greet() const {
        std::cout << "Hi, I am " << name << " and I am " << age << " years old." << std::endl;
    }
};

In [ ]:
// Instantiate and use the Person class
Person alice("Alice", 30);
alice.greet();

std::cout << "Name: " << alice.getName() << std::endl;
std::cout << "Age:  " << alice.getAge()  << std::endl;

// alice.age = -999;  // COMPILE ERROR: age is private!
// Expected output:
// Hi, I am Alice and I am 30 years old.
// Name: Alice
// Age:  30

## Why OOP Matters

Let us compare two approaches to modelling a shape: the C-style way (struct + free functions) and the C++ OOP way (class with methods).

Notice how the OOP version is:
- **Safer** — data is protected from accidental modification.
- **More readable** — `circle.getArea()` is clearer than `shapeGetArea(&circle)`.
- **More extensible** — you can add a `Rectangle` class that shares the same interface.

In [ ]:
#include <iostream>

// === C-style approach ===
struct CircleC {
    double radius;
};

double circleArea(const CircleC *c) {
    return 3.14159 * c->radius * c->radius;
}

CircleC c1;
c1.radius = 5.0;
std::cout << "[C-style] Area = " << circleArea(&c1) << std::endl;
// Expected output: [C-style] Area = 78.5397...

In [ ]:
#include <iostream>

// === OOP approach ===
class Circle {
private:
    double radius;

public:
    Circle(double r) {
        radius = r;
    }

    double getArea() const {
        return 3.14159 * radius * radius;
    }

    double getRadius() const {
        return radius;
    }
};

Circle c2(5.0);
std::cout << "[OOP] Area = " << c2.getArea() << std::endl;
// c2.radius = -1.0;  // COMPILE ERROR: radius is private!
// Expected output: [OOP] Area = 78.5397...

**Exercise 1:** Write a class called `Rectangle` with private members `width` and `height` (both `double`), a constructor that takes width and height, and a method `getArea()` that returns `width * height`. Also add a method `getPerimeter()` that returns `2 * (width + height)`. Instantiate it with width=4.0 and height=6.0 and print the area and perimeter.

Expected output:
```
Area: 24
Perimeter: 20
```

In [ ]:
// Your code here

## A More Detailed Class: Car

Let us look at a slightly richer example that shows private data, a constructor, multiple methods, and demonstrates the safety that encapsulation provides.

In [ ]:
#include <iostream>
#include <string>

class Car {
private:
    std::string brand;
    int         year;
    double      fuelLevel;   // 0.0 to 1.0 (fraction of full tank)

public:
    Car(const std::string &b, int y) {
        brand     = b;
        year      = y;
        fuelLevel = 1.0;  // start with a full tank
    }

    std::string getBrand() const { return brand; }
    int         getYear()  const { return year;  }
    double      getFuel()  const { return fuelLevel; }

    void drive(double distance) {
        double consumed = distance * 0.05;  // 5% per unit of distance
        if (consumed > fuelLevel) {
            std::cout << "Not enough fuel!" << std::endl;
        } else {
            fuelLevel -= consumed;
            std::cout << "Drove " << distance << " km. Fuel remaining: "
                      << (fuelLevel * 100) << "%" << std::endl;
        }
    }

    void refuel() {
        fuelLevel = 1.0;
        std::cout << "Tank refuelled." << std::endl;
    }
};

In [ ]:
Car myCar("Toyota", 2019);
std::cout << myCar.getBrand() << " (" << myCar.getYear() << ")" << std::endl;

myCar.drive(10.0);
myCar.drive(5.0);
myCar.refuel();
// Expected output:
// Toyota (2019)
// Drove 10 km. Fuel remaining: 50%
// Drove 5 km. Fuel remaining: 75%  <- after refuel, but let's re-drive
// Tank refuelled.

## Modern C++ (C++11 and Beyond)

C++11 brought a large number of language improvements. You will encounter these in real-world C++ code, so here is a brief preview. **This course focuses on C++98**, but it is good to know these exist.

Key C++11+ additions:

- **`auto` type inference** — let the compiler deduce the type: `auto x = 5;` instead of `int x = 5;`
- **`nullptr`** — a type-safe null pointer constant, replacing `NULL` and `0`
- **Range-based for loops** — `for (int val : myVector)` instead of index-based loops
- **Lambdas** — anonymous functions: `[](int x) { return x * 2; }`
- **Smart pointers** — `std::unique_ptr` and `std::shared_ptr` manage memory automatically
- **Move semantics** — efficient transfer of resources without copying
- **Uniform initialisation** — `MyClass obj{arg1, arg2}` works everywhere
- **`std::vector`, `std::array`** (already in C++98, but improved)

Here is a small C++11 preview — just to show how much cleaner some things become:

In [ ]:
#include <iostream>
#include <vector>
#include <string>

// C++11: auto and range-based for
std::vector<std::string> names;
names.push_back("Alice");
names.push_back("Bob");
names.push_back("Carol");

// C++98 way:
for (int i = 0; i < (int)names.size(); i++) {
    std::cout << "[C++98] " << names[i] << std::endl;
}

// C++11 way:
for (auto &n : names) {
    std::cout << "[C++11] " << n << std::endl;
}
// Both produce the same output — the C++11 version is more concise.